In [ ]:
import msprime
import tskit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
import kmapper as km
from kmapper.adapter import to_networkx
from kmapper.plotlyviz import plotlyviz
from ripser import Rips
from persim import plot_diagrams
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from funcs.simulation import simulate_one_pulse, simulate_two_pulses

from funcs.tract_processing import get_tracts_dict

from funcs.tract_extraction import get_population_tracts_dataframe

from funcs.bio_summary import build_individual_biology

from funcs.features import build_binary_matrix

from funcs.filters import compute_filter

from funcs.mapper_analysis import mapper_summary

from funcs.persistent_homology import persistence_summary

from parameters import params, t_second_wave, t, Ne


In [ ]:
w_size=1000
M=params['chrom_length']//w_size

Фильтры для Mapper

In [ ]:
filter_functions=pd.DataFrame({
    'filter_id':[1,2,3,4,5,6],
    'filter_name':[
        'MDS1',
        'distance_to_medoid',
        'mean_distance',
        'introgressed_fraction',
        'tract_count',
        'mean_tract_length'
    ],
    'filter_dim':[1,1,1,1,1,1]
})

Симуляция двух сценариев: один / две независимые волны интрорегрессии


In [ ]:
one_ts=simulate_one_pulse(42, 0.1)
two_ts=simulate_two_pulses(42,prop_first=0.05,prop_second=0.05)

Извлечение introgressed tracts

In [ ]:
one_tracts=get_tracts_dict(one_ts, params['n_eu'], [t['t_nd_migration']])
two_tracts=get_tracts_dict(two_ts,params['n_eu'],[t['t_nd_migration'],t_second_wave])

Построение бинарных матриц признаков: (строки = individuals, столбцы = genomic windows)

In [ ]:
Xa=build_binary_matrix(one_tracts, params['n_eu'], w_size, M)
Xb=build_binary_matrix(two_tracts, params['n_eu'], w_size, M) 

Cоздание "desert" сценария для one-pulse, зануляем участок генома